# Tests logiciels

Les tests vérifient le comportement du code avant son intégration.
Ce notebook rappelle les niveaux de tests usuels, montre un exemple pytest avec fixture, puis écrit les tests Kedro du pipeline `loading`.
Les tests portent sur la fonction du node, puis sur l'exécution du pipeline avec un catalogue dédié.
Les tests du modèle lui-même sont traités dans le notebook suivant.

## Niveaux de tests

Un test unitaire vérifie une portion atomique, en général une fonction. Il est rapide à écrire.
Un test de régression vérifie qu'une évolution ne réintroduit pas un bug déjà corrigé.
Un test d'intégration vérifie qu'une fonctionnalité s'assemble avec le reste de l'application. Ces écarts sont les plus difficiles à anticiper.

Sur une branche Git, trois règles s'appliquent.
Une branche est fusionnée lorsque les tests passent.
Chaque fonctionnalité nouvelle a un test.
Chaque correction de bug a un test, exécuté sur la correction.

Pour un modèle, métriques, courbes (PR, ROC, calibration, densité) et audit (PDP, valeurs de Shapley) constituent l'évaluation.
Les sous-populations de faux positifs et de faux négatifs à forte confiance en font partie.
Cette évaluation ne suffit pas à garantir le comportement attendu : le test de modèle ajoute des contrôles explicites.
Deux pratiques vont avec ces contrôles : conserver hyperparamètres, échantillons et modèle entraîné ; promouvoir en production un modèle qui dépasse les performances en place, ou un seuil fixé.

## Exemple pytest avec fixture

pytest découvre les fichiers `test*` et, dans chaque fichier, les fonctions `test*`.
Le décorateur `@pytest.fixture` définit une valeur réservée aux tests.
pytest injecte le retour de la fixture comme argument des fonctions de test. Le reste du programme ne charge pas ces données.

L'exemple ci-dessous vérifie `argmin` et `argmax` sur la même liste.
L'option `-v` détaille chaque test.

Les tests unitaires s'exécutent une fois le code envoyé vers le dépôt.
Ils restent hors du pre-commit : ce crochet doit rester court, et les tests de modèles peuvent être longs.

In [ ]:
%%writefile /tmp/pytest_2.py
import pytest

@pytest.fixture
def test_data():
    return [5, 8, 2, 9, 6, 3]

def argmin(liste):
    if len(liste) == 0:
        return None
    
    idx_min = 0
    value_min = liste[0]
    for i, x in enumerate(liste):
        if x < value_min:
            value_min = x
            idx_min = i
    return idx_min

def argmax(liste):
    if len(liste) == 0:
        return None
    
    idx_max = 0
    value_max = liste[0]
    for i, x in enumerate(liste):
        if x > value_max:
            value_max = x
            idx_max = i
    return idx_max

def test_argmin(test_data):
    assert argmin(test_data) == 2
    
def test_argmax(test_data):
    assert argmax(test_data) == 3 

In [ ]:
!pytest /tmp/pytest_2.py -v

## Tests Kedro du pipeline `loading`

Les tests reprennent l'arborescence du code : créer `src/tests/pipelines/loading/`.
Kedro fournit `src/tests/test_run.py` comme exemple, et la commande `kedro test`.

Avant les tests, déposer des données de test dans le bucket.
Sous `primary/`, créer le dossier `data-test.csv/` et y copier deux CSV déjà présents dans `data.csv/`.

Installer les dépendances de test du projet, qui incluent pytest.

### Tests des nodes

`conftest.py` regroupe les fixtures du dossier. pytest le charge avant les tests.
`scope="module"` construit chaque fixture une fois par module de test.
Les noms `project_id` et `primary_folder` sont les arguments de `test_load_csv_from_bucket`.

Le catalogue `catalog_test` sert au test de pipeline. Ses clés reprennent les entrées du node : `params:gcp_project_id` et `params:gcs_primary_folder`. La sortie du node est `primary`.
`MemoryDataset` garde ces valeurs en mémoire, sans écrire dans le catalogue du projet.

Le test de node vérifie le type, le nombre de colonnes (16) et la présence de `purchased`.

### Test du pipeline

`SequentialRunner` exécute le pipeline dans l'ordre, sur `catalog_test`.
Les assertions portent sur la sortie `primary`, avec les mêmes contrôles que le test de node.
Ce pipeline n'a qu'un node : les deux tests se rejoignent.